# Exercise 2
Today we are going to continue to work on point clouds.
We will work on clustering point clouds. That enables us to segment them.

In [76]:
import numpy as np
import open3d as o3d
import copy
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import os

os.environ["XDG_SESSION_TYPE"] = "x11"

In [77]:
def create_moebius_mesh(twists=1, radius=1.0, width=0.3, n_u=200, n_v=20):
    """Create a Möbius strip mesh (replacement for missing create_moebius)"""
    us = np.linspace(0, 2 * np.pi, n_u, endpoint=False)
    vs = np.linspace(-width, width, n_v)
    verts = []
    for u in us:
        cu, su = np.cos(u), np.sin(u)
        t = 0.5 * twists * u
        ct, st = np.cos(t), np.sin(t)
        for v in vs:
            x = (radius + v * ct) * cu
            y = (radius + v * ct) * su
            z = v * st
            verts.append([x, y, z])
    verts = np.array(verts, dtype=np.float64)

    tris = []
    for i in range(n_u):
        ni = (i + 1) % n_u
        for j in range(n_v - 1):
            a = i * n_v + j
            b = ni * n_v + j
            c = i * n_v + (j + 1)
            d = ni * n_v + (j + 1)
            tris.append([a, b, c])
            tris.append([b, d, c])

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(verts)
    mesh.triangles = o3d.utility.Vector3iVector(np.array(tris, dtype=np.int32))
    mesh.compute_vertex_normals()
    return mesh


In [78]:
def draw_labels_on_model(pcl,labels):
    cmap = plt.get_cmap("tab20")
    pcl_temp = copy.deepcopy(pcl)
    max_label = labels.max()
    print("%s has %d clusters" % (pcl_name, max_label + 1))
    colors = cmap(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0
    pcl_temp.colors = o3d.utility.Vector3dVector(colors[:, :3])
    o3d.visualization.draw_geometries([pcl_temp])



## K-means on a cube
We created a point cloud using `open3d`.
Our goal is to segment each side using k-means.

In [79]:
pcl_name = 'Cube'
density = 1e4 # density of sample points to create
pcl = o3d.geometry.TriangleMesh.create_box().sample_points_uniformly(int(density))
eps = 0.4
print("%s has %d points" % (pcl_name, np.asarray(pcl.points).shape[0]))
o3d.visualization.draw_geometries([pcl])

Cube has 10000 points


If we just use k-means out of the box with the point cloud, we will get what just has been visualized.

Note: Using the '+' and '-' keys in the viewer will increase/decrease the size of the points.

In [80]:
km = KMeans(n_clusters=6, init='random',
            n_init=10, max_iter=300, tol=1e-04, random_state=0)

# Get the points from the pointcloud as nparray
xyz = np.asarray(pcl.points)
labels = km.fit_predict(xyz)
draw_labels_on_model(pcl, labels)

Cube has 6 clusters


We can see that we get six clusters, but they do not span a side.

We try again, but this time we instead use the normals of the cube as input for k-means.

The normals for each plane should be parallel with the other normals from said plane.

In [81]:
# Estimate normals
pcl.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

# Get normals as array
normals = np.asarray(pcl.normals)

# K-means on normals only
km_normals = KMeans(n_clusters=6, init='random', n_init=10, max_iter=300, random_state=0)
labels_normals = km_normals.fit_predict(normals)

print("K-means with normals only:")
draw_labels_on_model(pcl, labels_normals)

K-means with normals only:
Cube has 6 clusters


This still does not work, opposite sides will also have normals that point the other way ($\vec{n}$ and $-\vec{n}$).

So, to combat this we can attempt to use the xyz coordinates and the normals.

## More exercises

### A) K-means continued.

Combine the point cloud points (xyz) with the normals and do k-means.

```xyz_n = np.concatenate((xyz, normals), axis=1)```

Do you get better clusters?
Why would adding the normals help?

### B) 
Try weighting either the points or normals by scaling them by some factor. Can this perfectly segment each of the faces of the cube?
### C)
Try to cluster all the different shapes using k means.
```{Python}
d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_moebius(twists=1).translate(
    (0, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_moebius(twists=2).translate(
    (d, -d, 0))
mesh.sample_points_uniformly(int(1e5)), 0.5
```

### D)
Now try segmenting a different point cloud located at `pointclouds/fragment.ply`
Are you able to cluster the point cloud?

Which features could be useful to segment this point cloud?
- fpfh features?
- xyz
- normals 
- colors

Are you able to get clusters that make sense? Why?

### E)
Use the built-in `cluster_dbscan` algorithm.
Tweak the parameters and see what you get out.

Attempt on the combined figures and on `fragment.ply`
```{Python}
#eps (float) – Density parameter that is used to find neighbouring points.
eps = 0.02

#min_points (int) – Minimum number of points to form a cluster.
min_points = 10

labels = np.array(pcl.cluster_dbscan(eps=eps, min_points=min_points, print_progress=True))
```

## Exercise A: Combine XYZ + Normals

In [82]:
print("=== Exercise A: XYZ + Normals ===")

# Combine xyz and normals
xyz_n = np.concatenate((xyz, normals), axis=1)

# K-means on combined features
km_combined = KMeans(n_clusters=6, init='random', n_init=10, max_iter=300, random_state=0)
labels_combined = km_combined.fit_predict(xyz_n)

print("K-means with xyz + normals:")
draw_labels_on_model(pcl, labels_combined)

print("\nWhy adding normals helps:")
print("- Normals encode surface orientation")
print("- Each face has unique position + orientation")
print("→ Better separation than xyz alone")

=== Exercise A: XYZ + Normals ===
K-means with xyz + normals:
Cube has 6 clusters

Why adding normals helps:
- Normals encode surface orientation
- Each face has unique position + orientation
→ Better separation than xyz alone


## Exercise B: Weighted Features

In [83]:
print("=== Exercise B: Weighted Features ===")

# Scale normals to give them more importance
normal_weight = 5.0

xyz_scaled = xyz.copy()
normals_scaled = normals * normal_weight
xyz_n_weighted = np.concatenate((xyz_scaled, normals_scaled), axis=1)

# K-means with weighted features
km_weighted = KMeans(n_clusters=6, init='random', n_init=10, max_iter=300, random_state=0)
labels_weighted = km_weighted.fit_predict(xyz_n_weighted)

print(f"K-means with normals weighted by {normal_weight}:")
draw_labels_on_model(pcl, labels_weighted)

=== Exercise B: Weighted Features ===
K-means with normals weighted by 5.0:
Cube has 6 clusters


## Exercise C: Multiple Shapes

In [84]:
print("=== Exercise C: Clustering Multiple Shapes ===")

# Create combined mesh
d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += create_moebius_mesh(twists=1, radius=1.0, width=0.3).translate((0, -d, 0))
mesh += create_moebius_mesh(twists=2, radius=1.0, width=0.3).translate((d, -d, 0))

pcl_shapes = mesh.sample_points_uniformly(int(1e5))
pcl_name = 'Multiple Shapes'

# Show original
o3d.visualization.draw_geometries([pcl_shapes])

# K-means on xyz
xyz_shapes = np.asarray(pcl_shapes.points)
km_shapes = KMeans(n_clusters=6, random_state=0)
labels_shapes = km_shapes.fit_predict(xyz_shapes)

print("Clustering 6 shapes with xyz:")
draw_labels_on_model(pcl_shapes, labels_shapes)

=== Exercise C: Clustering Multiple Shapes ===
Clustering 6 shapes with xyz:
Multiple Shapes has 6 clusters


## Exercise D: Fragment Point Cloud

In [85]:
print("=== Exercise D: Fragment Clustering ===")

# Try to load fragment, if not available create synthetic data
import os
if os.path.exists("pointclouds/fragment.ply"):
    pcl_fragment = o3d.io.read_point_cloud("pointclouds/fragment.ply")
else:
    print("fragment.ply not found, creating synthetic multi-object scene")
    # Create synthetic scene with multiple objects
    mesh_frag = o3d.geometry.TriangleMesh.create_sphere(radius=0.5).translate((-1, 0, 0))
    mesh_frag += o3d.geometry.TriangleMesh.create_box(width=0.8, height=0.8, depth=0.8).translate((0.5, -0.4, 0))
    mesh_frag += o3d.geometry.TriangleMesh.create_cylinder(radius=0.3, height=1.0).translate((0, 1.2, 0))
    mesh_frag.paint_uniform_color([0.7, 0.7, 0.7])
    pcl_fragment = mesh_frag.sample_points_uniformly(20000)
    # Add color variation
    colors = np.asarray(pcl_fragment.colors)
    pts = np.asarray(pcl_fragment.points)
    colors[:, 0] = (pts[:, 0] + 2) / 4  # R based on x
    colors[:, 1] = (pts[:, 1] + 2) / 4  # G based on y
    colors[:, 2] = (pts[:, 2] + 1) / 2  # B based on z
    pcl_fragment.colors = o3d.utility.Vector3dVector(colors)

pcl_name = 'Fragment'

# Check if point cloud has points
if len(pcl_fragment.points) == 0:
    print("ERROR: Point cloud is empty!")
else:
    o3d.visualization.draw_geometries([pcl_fragment])

    # Get features
    xyz_frag = np.asarray(pcl_fragment.points)
    colors_frag = np.asarray(pcl_fragment.colors)

    # Estimate normals
    pcl_fragment.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    normals_frag = np.asarray(pcl_fragment.normals)

    # Try different features
    print("\n1. XYZ only:")
    labels_xyz = KMeans(n_clusters=3, random_state=0).fit_predict(xyz_frag)
    draw_labels_on_model(pcl_fragment, labels_xyz)

    print("\n2. XYZ + Normals:")
    xyz_n_frag = np.concatenate((xyz_frag, normals_frag * 2), axis=1)
    labels_xyzn = KMeans(n_clusters=3, random_state=0).fit_predict(xyz_n_frag)
    draw_labels_on_model(pcl_fragment, labels_xyzn)

    print("\n3. XYZ + Colors:")
    xyz_c_frag = np.concatenate((xyz_frag, colors_frag * 3), axis=1)
    labels_xyzc = KMeans(n_clusters=3, random_state=0).fit_predict(xyz_c_frag)
    draw_labels_on_model(pcl_fragment, labels_xyzc)

    print("\nBest features: Colors work well for textured objects")

=== Exercise D: Fragment Clustering ===
fragment.ply not found, creating synthetic multi-object scene

1. XYZ only:
Fragment has 3 clusters

2. XYZ + Normals:
Fragment has 3 clusters

3. XYZ + Colors:
Fragment has 3 clusters

Best features: Colors work well for textured objects


## Exercise E: DBSCAN

In [86]:
print("=== Exercise E: DBSCAN ===")

# DBSCAN on cube
print("\n1. DBSCAN on Cube:")
pcl_name = 'Cube'
labels_db = np.array(pcl.cluster_dbscan(eps=0.02, min_points=10, print_progress=True))
draw_labels_on_model(pcl, labels_db)

# DBSCAN on shapes
print("\n2. DBSCAN on Shapes:")
pcl_name = 'Multiple Shapes'
labels_db_shapes = np.array(pcl_shapes.cluster_dbscan(eps=0.1, min_points=50, print_progress=True))
draw_labels_on_model(pcl_shapes, labels_db_shapes)

# DBSCAN on fragment (if available)
if len(pcl_fragment.points) > 0:
    print("\n3. DBSCAN on Fragment:")
    pcl_name = 'Fragment'
    labels_db_frag = np.array(pcl_fragment.cluster_dbscan(eps=0.05, min_points=20, print_progress=True))
    draw_labels_on_model(pcl_fragment, labels_db_frag)
else:
    print("\n3. DBSCAN on Fragment: Skipped (no data)")

print("\nDBSCAN vs K-means:")
print("- DBSCAN: Arbitrary shapes, handles noise")
print("- K-means: Fixed clusters, faster")


=== Exercise E: DBSCAN ===

1. DBSCAN on Cube:
Cube has 4 clusters
Precompute neighbors.[========================================] 100%

2. DBSCAN on Shapes:
Precompute neighbors.[========================================] 100%
Multiple Shapes has 326 clusters==========>        ] 77%

3. DBSCAN on Fragment:
Precompute neighbors.[========================================] 100%
Clustering[========================>     Fragment has 152 clusters
Clustering[=================================>      ] 82%
DBSCAN vs K-means:
- DBSCAN: Arbitrary shapes, handles noise
- K-means: Fixed clusters, faster
